# oma v2: `pi` 的对话框与按钮组件, 从 Jupyter 驱动

每个 `%%js` 单元经 CDP(**127.0.0.1:9229**)在 `node v2/demo.js repl` 里执行, 那里有全局的
`pi`(pi.js)、`sess`(menu.js 的 Session)和 `oma`(`setTheme` / `CFG` / `FRAMES` / `PRESETS` / `gfx`)。

* 先在终端里跑 `node v2/demo.js repl`(可加 `port=9230`、`color.preset=pink`、`cfg.btnH=3` 等); `%%js 9230` 连别的端口。
* 单元是 REPL 语义: 顶层 `await`, `let` / `const` 可重复声明。
* `await` 对话框的单元在终端里答完才结束(Esc → `null`)。下面的输出是用脚本按键跑出来的。
* 终端被 node 的 `Debugger ending…` 弄花时按 **Ctrl-L**(或 `sess.repaint()`)。

和 goal-v1.ipynb 的区别: 这里只有渲染侧(pi.js + menu.js), 没有 claude-p 宿主的 `pi.sh` / `` pi`…` `` /
`pi.tui` / `pi.on` 等; 对话框按 v2 的 API 写:

```js
const [c, c1] = pi.onEmit({ title, body, footer, ...state })   // c 读, c1.x(v) / c1({...}) 写, 同一 microtask 的写入合并
pi.dialog({ init: c /* 或不变的 { title, body, footer } */, eachRow, timeout, or, update, fadeX, drop: { key: items } },
          { key: up => label | void | Promise })          // up=false 渲染: 返回标签; up=true 按下: 返回字符串 = 不关
// resolve 为按下的键名(Esc = null); 句柄: .ret(v) 从外部关闭, .drop(key, items) 程序化地在按钮下面弹出下拉列表
// drop 里的键是下拉框: 标签自动追加 ' ▼', 按下即弹出列表(焦点移过去), 选中后调用 bs[key](选中项)
```

In [1]:
# %%js [port] [timeout]: evaluate the cell in `node v2/demo.js repl`, where the globals
# `pi` / `sess` / `oma` live, over CDP (default port 9229).
# Stdlib only: a minimal WebSocket client, so the kernel needs no packages.
# One connection per port is kept open: every disconnect makes node print "Debugger ending on …"
# straight onto the TUI (Ctrl-L or sess.repaint() redraws it).
import base64, json, os, socket, struct, urllib.request
from urllib.parse import urlparse

def _ws_connect(url, timeout):
    u = urlparse(url)
    s = socket.create_connection((u.hostname, u.port or 80), timeout=timeout)
    key = base64.b64encode(os.urandom(16)).decode()
    s.sendall((f'GET {u.path} HTTP/1.1\r\nHost: {u.hostname}:{u.port}\r\nUpgrade: websocket\r\n'
               f'Connection: Upgrade\r\nSec-WebSocket-Key: {key}\r\nSec-WebSocket-Version: 13\r\n\r\n').encode())
    head = b''
    while b'\r\n\r\n' not in head:
        chunk = s.recv(4096)
        if not chunk:
            raise ConnectionError('inspector closed the connection')
        head += chunk
    if b' 101 ' not in head.split(b'\r\n', 1)[0]:
        raise ConnectionError(head.split(b'\r\n', 1)[0].decode())
    return s, head.split(b'\r\n\r\n', 1)[1]

def _ws_send(s, text):
    data = text.encode()
    n = len(data)
    hdr = bytes([0x81]) + (bytes([0x80 | n]) if n < 126 else
                           bytes([0x80 | 126]) + struct.pack('>H', n) if n < 65536 else
                           bytes([0x80 | 127]) + struct.pack('>Q', n))
    mask = os.urandom(4)
    s.sendall(hdr + mask + bytes(b ^ mask[i % 4] for i, b in enumerate(data)))

class _Reader:
    def __init__(self, s, buf):
        self.s, self.buf = s, buf
    def take(self, n):
        while len(self.buf) < n:
            chunk = self.s.recv(65536)
            if not chunk:
                raise ConnectionError('inspector closed the connection')
            self.buf += chunk
        out, self.buf = self.buf[:n], self.buf[n:]
        return out
    def message(self):
        parts = []
        while True:
            b0, b1 = self.take(2)
            n = b1 & 0x7f
            if n == 126: n = struct.unpack('>H', self.take(2))[0]
            elif n == 127: n = struct.unpack('>Q', self.take(8))[0]
            payload = self.take(n)
            op = b0 & 0x0f
            if op == 0x8: raise ConnectionError('inspector closed the connection')
            if op in (0x9, 0xA): continue  # ping / pong
            parts.append(payload)
            if b0 & 0x80: return b''.join(parts).decode()

_conns = {}  # port -> [socket, reader, last id]

def _conn(port, timeout):
    if port not in _conns:
        targets = json.load(urllib.request.urlopen(f'http://127.0.0.1:{port}/json/list', timeout=5))
        s, rest = _ws_connect(targets[0]['webSocketDebuggerUrl'], timeout)
        _conns[port] = [s, _Reader(s, rest), 0]
    return _conns[port]

def _drop(port):
    c = _conns.pop(port, None)
    if c: c[0].close()

def cdp_eval(code, port=9229, timeout=300):
    """Evaluate `code` in the demo (top-level await allowed); return its value."""
    for retry in (True, False):
        c, sent = None, False
        try:
            c = _conn(port, timeout)
            c[0].settimeout(timeout)
            c[2] += 1
            _ws_send(c[0], json.dumps({'id': c[2], 'method': 'Runtime.evaluate', 'params': {
                'expression': code, 'replMode': True, 'awaitPromise': True, 'returnByValue': True}}))
            sent = True
            while True:
                msg = json.loads(c[1].message())
                if msg.get('id') == c[2]:
                    break
            break
        except OSError:  # includes ConnectionError / timeouts: the connection is no longer usable
            _drop(port)
            if sent or not retry: raise  # never run a cell twice; a stale connection is retried once
    res = msg.get('result', {})
    if 'exceptionDetails' in res:
        d = res['exceptionDetails']
        raise RuntimeError((d.get('exception') or {}).get('description') or d.get('text'))
    v = res.get('result', {})
    return v.get('value', v.get('description', v.get('type')))

try:
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def js(line, cell):
        args = line.split()
        return cdp_eval(cell, int(args[0]) if args else 9229, float(args[1]) if len(args) > 1 else 300)
except ImportError:
    pass  # plain Python: call cdp_eval(code) directly

In [2]:
%%js
({ pi: Object.keys(pi), size: `${process.stdout.columns}x${process.stdout.rows}`, gfx: oma.gfx.mode, presets: Object.keys(oma.PRESETS), frames: oma.FRAMES })

{'pi': ['dialog', 'inlineKbd', 'onEmit', 'if404', 'omarchy', 'notify', 'arg'],
 'size': '227x42',
 'gfx': 'direct',
 'presets': ['green', 'pink'],
 'frames': {'heavy': '┏━┓┃ ┃┗━┛', 'light': '┌─┐│ │└─┘', 'round': '╭─╮│ │╰─╯'}}

## 1. 对话框(goal-v1 §1 按新 API 重写)

| goal-v1 | v2 |
|---|---|
| `pi.dialog({ title, body, ...buttons })` | `pi.dialog({ init }, buttons)` |
| 按钮 `() => label`, 键名 = 初始标签 | `up => label`: 渲染时也调用, 键名固定 |
| `.text({ body })` | `c1.body(…)` / `c1({ … })` |
| `.relabel(i, label)` | 写 State, 依赖它的按钮自己重新渲染 |
| `hints` | 还没有(按钮渲染时不知道自己有没有焦点); 1.9 / 2.5 在按下时写 footer |

### 1.1 OK / Cancel(`init` 可以是不变的对象)

In [4]:
%%js
await pi.dialog({ init: { title: 'Hello', body: 'from Jupyter' } }, { OK() {}, Cancel() {} })

### 1.2 一直开着的网格: 按下返回字符串 = 不关(`eachRow: [3, 3, 3, 3]`)

In [6]:
%%js
const [calc, calc1] = pi.onEmit({ title: 'add', body: ' ', footer: '= closes', expr: '' })
const digits = ['7', '8', '9', '4', '5', '6', '1', '2', '3', '0', '+', '=']
// 键名不能是 '7' 这种整数样的字符串(JS 对象会把它们排到最前、按数值升序), 用 'n7', 标签另外返回
await pi.dialog({ init: calc, eachRow: [3, 3, 3, 3] }, Object.fromEntries(digits.map(k => ['n' + k, up => {
  if (up && k === '=') return              // void: 关
  if (up) calc1({ expr: calc.expr + k, body: calc.expr + k })
  return k                                 // 渲染: 标签; 按下: 字符串 = 不关
}])))
calc.expr.split('+').reduce((sum, n) => sum + Number(n || 0), 0)

9

### 1.3 从数据生成的竖排列表(`eachRow: 'vertical'`)

In [6]:
%%js
const { execFileSync } = process.getBuiltinModule('node:child_process')
const branches = execFileSync('git', ['branch', '--format=%(refname:short)'], { cwd: process.cwd(), encoding: 'utf8' })
  .split('\n').filter(Boolean).slice(0, 8)
const branch = await pi.dialog({ init: { title: '$ Git branches', footer: 'Enter shows it in the status line' }, eachRow: 'vertical' },
  Object.fromEntries(branches.map(b => [b, () => {}])))
if (branch) sess.status(`branch: ${branch}`)
branch

'main'

### 1.4 单选: 写 State, 兄弟按钮跟着换点(`eachRow: [4, 1]`)

`update` 默认 `'oldnew'`: State 变化只重新渲染上一个 / 当前焦点, 以及标签是单选语法(`●` `○` `( )` `(*)`)的按钮, 所以这里不用 `'full'`。

In [9]:
%%js
const [shirt, shirt1] = pi.onEmit({ title: 'T-shirt', body: 'one button per value', size: 'M' })
const size = v => up => (up && shirt1.size(v), `${shirt.size === v ? '●' : '○'} ${v}`)
await pi.dialog({ init: shirt, eachRow: [4, 1] }, { S: size('S'), M: size('M'), L: size('L'), XL: size('XL'), Done() {} })
shirt.size

'XL'

### 1.5 勾选(开头 `[x]` / `[ ]`)

In [8]:
%%js
const on = { lint: true, tests: false, docs: false }
const box = k => up => (up && (on[k] = !on[k]), `[${on[k] ? 'x' : ' '}] ${k}`)
await pi.dialog({ init: { title: 'Before commit' }, eachRow: [3, 1] }, { ...Object.fromEntries(Object.keys(on).map(k => [k, box(k)])), Done() {} })
on

{'lint': False, 'tests': True, 'docs': False}

### 1.6 实时正文(`c.setTimeout`; 对话框关闭时定时器一起清掉)

In [10]:
%%js
const [live, live1] = pi.onEmit({ title: 'Live', body: 'tick 0', footer: '', n: 0 })
live.setTimeout(-500, () => live1({ n: live.n + 1, body: `tick ${live.n + 1}`, footer: new Date().toLocaleTimeString() }))
await pi.dialog({ init: live }, { Stop() {} })
live.n

3

### 1.7 非模态 toast: 右上角, 到点按下 `or`, 任何交互取消自动选择

In [16]:
%%js
await pi.dialog({ init: { title: 'Saved', body: 'auto-answers in' }, timeout: 3000, or: 'Keep' }, { Keep() {}, Undo() {} })

'Keep'

### 1.8 `update: 'oldnew'`: State 每 0.4 s 变一次, 只有上一个 / 当前焦点在重新渲染(标签 = 渲染次数); 换成 `'full'` 四个都会涨

In [31]:
%%js
const runs = {}
const count = k => up => up ? undefined : `${k} ×${runs[k] = (runs[k] ?? 0) + 1}`
const [t, t1] = pi.onEmit({ title: 'update: oldnew', n: 0 })
t.setTimeout(-400, () => t1.n(t.n + 1))
await pi.dialog({ init: t }, { A: count('A'), B: count('B'), C: count('C'), D: count('D') })
runs

{'A': 6, 'B': 4, 'C': 1, 'D': 1}

## 2. 新的按钮样式: 都只认标签文字

| 标签 | 画法 |
|---|---|
| `[...]` | spacer: 不可见、不可聚焦(←/→/↑/↓ 跳过), 平分该行剩余宽度 |
| `eachRow` 里的 `-n` | n 个按钮的分段组: 撑满对话框、平分宽度、无间隔; 组底色 `groupBg`, `●` 段 = 选中段(`btnBg` + 强调色, 图标不画); **键盘移到哪段就按下哪段**(鼠标悬停不算) |
| `text ▼` | 下拉框: 撑满该行, `▼` 贴右。用 `drop: { key: items }` 声明时 `▼` 自动追加, 按下即在它下面弹出列表并抢走焦点; 选中后调用 `bs[key](选中项)`; `~` 开头的项禁用。列表开着时点别处 = Esc |
| `text [x]` / `text [ ]` | 开关(标记在结尾; 在开头仍是勾选图标): 撑满该行, 直角, 贴右。有 gra.py 时按 svg 稿画(viewBox 42x22, 高 = 1 行): 轨道 42x22(关: `track` 填充 + 1 单位 `trackEdge` 描边; 开: `trackOn` 整块), 滑块 16x16 在 x=3 / 23, 颜色 `knob`; 三块各自补间(位置 + 颜色)。无 gra.py 时是 4 格轨道 + 2 格滑块 |

撑满的行(含 spacer / 下拉框 / 开关, 或负数)不参与普通行的列对齐; ↑/↓ 取目标行里中心最近的按钮。

在所有窗口之外点击 = Esc: 对话框和下拉列表得 `null`(非模态 toast 不关), `pi.omarchy.menu` 得 `null`。

### 2.1 spacer: Reset 靠左, Cancel / Save 靠右

In [18]:
%%js
await pi.dialog({ init: { title: 'Settings', body: 'spacer [...] takes the free width' } },
  { Reset() {}, sp: () => '[...]', Cancel() {}, Save() {} })

'Cancel'

### 2.2 分段组(`eachRow: [-4, 1]`): →/← 直接选中, 不用按 Enter

In [32]:
%%js
const [tb, tb1] = pi.onEmit({ title: 'Button group', body: 'Toolbar is placed at the top.', footer: '←/→ move · Enter choose', pos: 'Top' })
const pos = v => up => (up && tb1({ pos: v, body: `Toolbar is placed at the ${v.toLowerCase()}.` }), `${tb.pos === v ? '●' : '○'} ${v}`)
await pi.dialog({ init: tb, eachRow: [-4, 1] }, { Top: pos('Top'), Right: pos('Right'), Bottom: pos('Bottom'), Left: pos('Left'), Done() {} })
tb.pos

'Bottom'

### 2.3 下拉框(`drop`): ` ▼` 自动追加, 按下即弹出; 打字过滤, `~` 项禁用(暗色、跳过), Esc 保持原值

In [3]:
%%js
const [ws, ws1] = pi.onEmit({ title: 'Combobox', body: 'Search a collection, then choose a matching option.',
                              footer: 'This setting is managed by your organization.', ws: 'Find a workspace…' })
const options = ['Personal workspace', 'Team workspace', '~Archived workspace', 'Sandbox']
await pi.dialog({ init: ws, eachRow: [1, 1], drop: { ws: options } }, {
  ws: v => (v && ws1.ws(v), ws.ws),       // 渲染时 v = false; 选中后 v = 选中项; 标签自动追加 ' ▼'
  OK() {},
})
ws.ws

'Team workspace'

### 2.4 开关(结尾 `[x]` / `[ ]`)

In [2]:
%%js
const sw = { '国内镜像': true, 'Auto update': false, 'Telemetry': false }
const toggle = k => up => (up && (sw[k] = !sw[k]), `${k} ${sw[k] ? '[x]' : '[ ]'}`)
await pi.dialog({ init: { title: 'Plugins', footer: 'Enter toggles · Esc closes' }, eachRow: 'vertical' },
  Object.fromEntries(Object.keys(sw).map(k => [k, toggle(k)])))
sw

{'国内镜像': True, 'Auto update': True, 'Telemetry': True}

### 2.4b 点外面 = Esc: 第一次点击关掉下拉列表(值不变), 第二次关掉对话框(得 `null`)

In [40]:
%%js
const [bl, bl1] = pi.onEmit({ title: 'blur', body: 'click outside to close', ws: 'Sandbox' })
const result = await pi.dialog({ init: bl, drop: { ws: ['Personal workspace', 'Sandbox'] } }, {
  ws: v => (v && bl1.ws(v), bl.ws),
  OK() {},
})
;({ result, ws: bl.ws })

{'result': None, 'ws': 'Sandbox'}

### 2.5 组合: 截图里的 “this session” 面板(分段组 ×3 + 勾选 + spacer, 标题由定时器更新)

In [48]:
%%js
const [ss, ss1] = pi.onEmit({ title: 'this session  ·  $0.00', footer: '', cost: 0,
                              model: 'opus', effort: 'medium', mode: 'auto', file: false, web: false, bash: true })
const hint = { file: 'Show or hide file edits', web: 'Allow web fetches', bash: 'Show or hide Bash tool output on screen' }
const radios = (g, vs) => Object.fromEntries(vs.map(v => [`${g}:${v}`, up => (up && ss1[g](v), `${ss[g] === v ? '●' : '○'} ${v}`)]))
const tick = k => up => (up && ss1({ [k]: !ss[k], footer: hint[k] }), `[${ss[k] ? 'x' : ' '}] ${k}`)
ss.setTimeout(-1000, () => ss1({ cost: ss.cost + 0.01, title: `this session  ·  $${(ss.cost + 0.01).toFixed(2)}` }))
const picked = await pi.dialog({ init: ss, eachRow: [-4, -5, -3, 3, 5] }, {
  ...radios('model', ['haiku', 'sonnet', 'opus', 'fable']),
  ...radios('effort', ['low', 'medium', 'high', 'xhigh', 'max']),
  ...radios('mode', ['plan', 'acc', 'auto']),
  file: tick('file'), web: tick('web'), bash: tick('bash'),
  CtrlO() {}, CtrlR() {}, sp: () => '[...]', Undo() {}, Copy() {},
})
;({ picked, model: ss.model, effort: ss.effort, mode: ss.mode, bash: ss.bash })

{'picked': 'Copy',
 'model': 'opus',
 'effort': 'high',
 'mode': 'plan',
 'bash': True}

## 3. 主题: 9-patch 边框 + 预设

`oma.setTheme({ preset, frame, ...颜色 })`: `preset` 先套整套(`green` 默认 / `pink` 圆角粉色 / `paper` 浅色, 开关取自 svg 稿 / `nord`), 同一次调用里其它键再覆盖它;
`frame` 是 `'heavy' | 'light' | 'round'` 或任意 9 个字符(左上 上 右上 / 左 中 右 / 左下 下 右下, 中间那格不画)。
命令行同样可用: `node v2/demo.js repl color.preset=pink color.frame=light`。新键: `groupBg`(分段组底色)、`track` `trackEdge` `trackOn` `knob`(开关: 关时轨道 / 关时描边 / 开时轨道 / 滑块)、`selText`(焦点按钮文字色, `null` = `selFg`)。

`pi.arg.focusHL`: `'color'` = 焦点按钮文字用 `selText`(没有则 `selFg`), `'#rrggbb'` = 用这个颜色, `''` = 不用(只在 gra.py 生效时才真的关掉, 否则焦点看不出来)。

In [49]:
%%js
const [th, th1] = pi.onEmit({ title: 'theme', body: 'preset / frame / focusHL apply live', preset: 'green', frame: 'heavy', hl: pi.arg.focusHL })
const radio = (k, v, apply, text = v) => up => {
  if (up) { apply(v); th1({ [k]: v, ...(k === 'preset' && { frame: oma.PRESETS[v].frame }) }) }
  return `${th[k] === v ? '●' : '○'} ${text}`
}
const theme = k => v => oma.setTheme({ [k]: v }), hl = v => { pi.arg.focusHL = v }
await pi.dialog({ init: th, eachRow: [-4, -3, -3, 1] }, {
  green: radio('preset', 'green', theme('preset')), pink: radio('preset', 'pink', theme('preset')),
  paper: radio('preset', 'paper', theme('preset')), nord: radio('preset', 'nord', theme('preset')),
  heavy: radio('frame', 'heavy', theme('frame')), light: radio('frame', 'light', theme('frame')), round: radio('frame', 'round', theme('frame')),
  hlOff: radio('hl', '', hl, "focusHL ''"), hlColor: radio('hl', 'color', hl, "'color'"), hlGold: radio('hl', '#ffd75f', hl, "'#ffd75f'"),
  Done() {},
})
;({ preset: th.preset, frame: th.frame, hl: th.hl })

{'preset': 'green', 'frame': 'heavy', 'hl': 'color'}

### 3.1 收尾: 回到默认主题

In [16]:
%%js
oma.setTheme({ preset: 'green' }); pi.arg.focusHL = 'color'; sess.repaint(); 'green'

'green'